In [5]:
# ==========================================
# OPTIMIZED PRICE PREDICTION MODEL (Best R²)
# ==========================================
# Install dependencies if needed:
# !pip install pandas numpy xgboost scikit-learn joblib

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
print("--- Step 1: Loading Data ---")
df = pd.read_csv('crop_price_dataset (1).csv')
df['month'] = pd.to_datetime(df['month'])
df = df.sort_values(by=['commodity_name', 'month']).reset_index(drop=True)
print(f"Data loaded. Shape: {df.shape}")

# 2. Advanced Feature Engineering (The key to high R²)
print("\n--- Step 2: Feature Engineering ---")
# Extract time features
df['year'] = df['month'].dt.year
df['month_num'] = df['month'].dt.month

# Create Lag Features (Previous months' prices)
# This gives the model memory of what happened previously
df['price_lag_1'] = df.groupby('commodity_name')['avg_modal_price'].shift(1)
df['price_lag_2'] = df.groupby('commodity_name')['avg_modal_price'].shift(2)
df['price_lag_3'] = df.groupby('commodity_name')['avg_modal_price'].shift(3)

# Create Rolling Mean (Average of last 3 months)
df['price_rolling_mean_3'] = df.groupby('commodity_name')['avg_modal_price'].transform(
    lambda x: x.rolling(window=3).mean()
)

# Drop the first 3 months of each crop because they don't have lag data
df = df.dropna().reset_index(drop=True)

# Encode categorical variables
le_commodity = LabelEncoder()
df['commodity_encoded'] = le_commodity.fit_transform(df['commodity_name'])
print("Feature Engineering complete.")

# 3. Feature Selection & Time-Based Split
print("\n--- Step 3: Data Splitting ---")
features = ['year', 'month_num', 'commodity_encoded', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_rolling_mean_3']
target = 'avg_modal_price'

X = df[features]
y = df[target]

# Time-based split (Train on older data, test on newest 20% of data)
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")

# 4. Model Training (Tuned XGBoost)
print("\n--- Step 4: Model Training ---")
model = xgb.XGBRegressor(
    n_estimators=500,       # Increased trees for better learning
    learning_rate=0.05,     # Lower learning rate for finer tuning
    max_depth=7,            # Deeper trees to capture complex patterns
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)
print("Model training complete.")

# 5. Evaluation (R² and Accuracy)
print("\n--- Step 5: Evaluation ---")
predictions = model.predict(X_test)

# Standard Metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

# MAPE to "Accuracy" Conversion
mape = np.mean(np.abs((y_test - predictions) / (y_test + 1e-9))) * 100
accuracy_percentage = 100 - mape

print(f"Price Prediction Model - MAE: {mae:.2f}")
print(f"Price Prediction Model - RMSE: {rmse:.2f}")
print(f"--> Price Prediction R² Score: {r2:.4f} (Closer to 1.0 is better)")
print(f"--> Price Prediction Overall Accuracy: {accuracy_percentage:.2f}%")

# 6. Save Model & Encoders
joblib.dump(model, 'price_model.pkl')
joblib.dump(le_commodity, 'le_commodity.pkl')
print("\nPrice Prediction Model and Encoders Saved Successfully.")

--- Step 1: Loading Data ---
Data loaded. Shape: (2810, 9)

--- Step 2: Feature Engineering ---
Feature Engineering complete.

--- Step 3: Data Splitting ---
Training set: (2209, 7), Testing set: (553, 7)

--- Step 4: Model Training ---
Model training complete.

--- Step 5: Evaluation ---
Price Prediction Model - MAE: 981.85
Price Prediction Model - RMSE: 2368.64
--> Price Prediction R² Score: 0.9042 (Closer to 1.0 is better)
--> Price Prediction Overall Accuracy: 92.91%

Price Prediction Model and Encoders Saved Successfully.
